# Phase 1: Explore the data

*Made with ❤️ by Calli*

**Where this fits in the pipeline:** `[look at the data]` → data → model → train → evaluate → predict

Before building anything, we look at the data. Every real ML project starts here, because surprises in the data (duplicates, junk files, one patient's cells everywhere) silently ruin a model later.

### How to run this on Kaggle
1. On Kaggle: **Create → New Notebook**, then **File → Import Notebook** and upload this file.
2. Right panel: **Add Input** → search `cell-images-for-detecting-malaria` (by *iarunava*) → **Add**.
3. No GPU needed for this notebook. Click **Run All**.
4. Copy the output of the **last cell** (the summary): Phase 2's patient split is built on it.

On a laptop, this notebook uses the fake dataset from `python -m src.make_fake_data` instead.

*This notebook is self-contained on purpose: it doesn't import from `src/`, so it runs on Kaggle without cloning the repo.*

## 1. Imports

- `hashlib` makes a "fingerprint" (hash) of each file's contents, used to find duplicates.
- `re` is regular expressions: a mini-language for matching text patterns, used to pull the patient code out of filenames.
- `pandas` keeps one row per image in a table, like a spreadsheet.

In [ ]:
import hashlib
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# Same seed every run, so the "random" sample grids below are the same each time.
RANDOM_SEED = 42

# The two class folder names. (Same as CLASS_NAMES in src/config.py.)
CLASS_NAMES = ["Uninfected", "Parasitized"]

## 2. Find the data folder

On Kaggle, attached datasets appear under `/kaggle/input/`. We **don't** hardcode a deep path like `/kaggle/input/.../cell_images/cell_images/Parasitized`, because the exact layout differs between dataset versions. We just give the top folder and search inside it.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")

if KAGGLE_INPUT.exists():
    DATA_DIR = KAGGLE_INPUT
else:
    # Running locally: use the fake dataset. The notebook may be started from
    # the project folder or from inside notebooks/, so try both.
    possible_fake_dirs = [Path("data/fake"), Path("../data/fake")]
    existing_fake_dirs = [folder for folder in possible_fake_dirs if folder.exists()]
    if not existing_fake_dirs:
        raise FileNotFoundError("No data found. Locally, run `python -m src.make_fake_data` first.")
    DATA_DIR = existing_fake_dirs[0]

print("Searching for data in:", DATA_DIR.resolve())

## 3. What files are there? (junk check)

We list **every** file and count them by extension. We expect `.png` images, plus some junk such as `Thumbs.db` (a thumbnail cache Windows creates). Only `.png` files are real data.

In [ ]:
all_files = [path for path in DATA_DIR.rglob("*") if path.is_file()]

extension_counts = Counter(path.suffix.lower() for path in all_files)
print(f"Total files found: {len(all_files)}")
for extension, count in extension_counts.most_common():
    shown_extension = extension if extension else "(no extension)"
    print(f"  {shown_extension:>15}: {count}")

junk_files = [path for path in all_files if path.suffix.lower() != ".png"]
print(f"\nJunk (non-.png) files: {len(junk_files)}. Examples:")
for path in junk_files[:5]:
    print("  ", path)

## 4. Build a table of all `.png` images

The **label** of an image is the name of the folder it sits in (`Parasitized` or `Uninfected`). We also record which folder **depth** it was found at, so we can see whether the nested duplicate `cell_images/cell_images/` folder exists.

In [ ]:
png_files = [path for path in all_files if path.suffix.lower() == ".png"]

rows = []
for path in png_files:
    folder_name = path.parent.name
    if folder_name not in CLASS_NAMES:
        continue  # a .png outside a class folder: not labelled, so skip it
    rows.append({
        "path": str(path),
        "filename": path.name,
        "label": folder_name,
        "folder": str(path.parent.relative_to(DATA_DIR)),
    })

images = pd.DataFrame(rows)
skipped = len(png_files) - len(images)
print(f"{len(images)} labelled .png files ({skipped} .png files skipped: not inside a class folder)")
print("\nImages per folder (two copies of each class folder = nested duplicate):")
print(images.groupby(["folder"]).size().to_string())

## 5. Find duplicate images

Two files with **identical contents** have the same *hash*: a short fingerprint calculated from every byte of the file. If any single byte differs, the hash is completely different.

Duplicates are dangerous: if the same image lands in both the training set and the test set, the model has already "seen the answer", and the test score becomes fake-good.

We also check the worst case: the **same image under both labels**. That would mean the dataset contradicts itself.

*(On the real dataset this reads about 55,000 small files: expect about a minute.)*

In [ ]:
def file_hash(path: str) -> str:
    # MD5 is fast and fine for spotting identical files (not for security).
    with open(path, "rb") as image_file:
        return hashlib.md5(image_file.read()).hexdigest()

images["hash"] = images["path"].apply(file_hash)

copies_per_hash = images["hash"].value_counts()
duplicated_hashes = copies_per_hash[copies_per_hash > 1]
print(f"Files:             {len(images)}")
print(f"Unique images:     {images['hash'].nunique()}")
print(f"Images with >1 copy: {len(duplicated_hashes)}")

# Do the duplicates share the same filename (= nested folder copy),
# or are they the same picture saved under different names?
names_per_hash = images.groupby("hash")["filename"].nunique()
renamed_duplicates = names_per_hash[(names_per_hash > 1)]
print(f"Duplicates saved under DIFFERENT filenames: {len(renamed_duplicates)}")

# Worst case: the same picture labelled both Parasitized and Uninfected.
labels_per_hash = images.groupby("hash")["label"].nunique()
conflicting_hashes = labels_per_hash[labels_per_hash > 1]
print(f"Same image under BOTH labels: {len(conflicting_hashes)}")

Now we keep **one** copy of each unique image. From here on, `unique_images` is our real dataset.

In [ ]:
unique_images = images.drop_duplicates(subset="hash", keep="first").reset_index(drop=True)
print(f"After removing duplicates: {len(unique_images)} images")

## 6. Class balance

Is it roughly 50/50? If one class were, say, 95% of the data, a lazy model could score 95% accuracy by always guessing that class. Balanced data makes accuracy a more honest number.

In [ ]:
class_counts = unique_images["label"].value_counts()
class_shares = (class_counts / len(unique_images) * 100).round(1)
print(pd.DataFrame({"images": class_counts, "percent": class_shares}).to_string())

## 7. Image sizes

A CNN needs every image to be the **same size**, so we will resize them all. Looking at the real sizes tells us what size makes sense: too small loses the parasite's detail, too large wastes computation.

`Image.open` only reads the file's header here, not the pixels, so this is quick.

In [ ]:
def image_width_height(path: str) -> tuple[int, int]:
    with Image.open(path) as image:
        return image.size  # (width, height)

sizes = unique_images["path"].apply(image_width_height)
unique_images["width"] = [size[0] for size in sizes]
unique_images["height"] = [size[1] for size in sizes]

print(unique_images[["width", "height"]].describe().round(1).to_string())

figure, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(unique_images["width"], bins=50)
axes[0].set_title("Image widths (pixels)")
axes[1].hist(unique_images["height"], bins=50)
axes[1].set_title("Image heights (pixels)")
plt.tight_layout()
plt.show()

## 8. Look at the cells

16 random cells per class. Try it yourself: **can you spot the infected ones by eye?** Infected cells usually show a dark purple stained blob (the parasite). The number above each cell is its original size.

In [ ]:
def show_random_cells(label: str, how_many: int = 16) -> None:
    rows_for_label = unique_images[unique_images["label"] == label]
    sample = rows_for_label.sample(n=min(how_many, len(rows_for_label)), random_state=RANDOM_SEED)

    figure, axes = plt.subplots(2, 8, figsize=(14, 5))
    for axis, (_, row) in zip(axes.flat, sample.iterrows()):
        axis.imshow(Image.open(row["path"]))
        axis.set_title(f"{row['width']}x{row['height']}", fontsize=8)
        axis.axis("off")
    figure.suptitle(label, fontsize=14)
    plt.tight_layout()
    plt.show()

for class_name in CLASS_NAMES:
    show_random_cells(class_name)

## 9. Filenames: look first, trust later

Many cells come from the **same patient's slide**. If one patient's cells end up in both training and test sets, the model can learn to recognise *that slide* (its stain colour, lighting) instead of learning malaria, and the test score lies. So in Phase 2 we split **by patient**.

The patient/slide code is believed to sit at the start of each filename. **Before trusting that, we print real filenames and look.**

In [ ]:
for class_name in CLASS_NAMES:
    names = unique_images.loc[unique_images["label"] == class_name, "filename"]
    print(f"--- {class_name}: 8 random filenames ---")
    for name in names.sample(n=min(8, len(names)), random_state=RANDOM_SEED):
        print("  ", name)

## 10. Parse the patient code

A filename like `C33P1thinF_IMG_20150619_114756a_cell_179.png` seems to start with a code (`C33P1`) followed by `thinF` (a *thin* blood *F*ilm, the kind of slide used).

The **regular expression** below means: *"take everything from the start of the name up to `thinF`, ignoring upper/lower case and any underscore in between."* Every filename that does **not** fit is printed, because a pattern we can't explain is a pattern we can't trust.

We don't yet know whether the patient is identified by the whole code (`C33P1`), by the `C` number, or by the `P` number, so we count all three.

In [ ]:
# ^       = start of the filename
# (.+?)   = capture the shortest possible run of characters: the code
# _?      = an optional underscore
# thinF   = the literal text "thinF" (case-insensitive because of re.IGNORECASE)
CODE_PATTERN = re.compile(r"^(.+?)_?thinF", re.IGNORECASE)

def extract_code(filename: str) -> str | None:
    match = CODE_PATTERN.match(filename)
    if match is None:
        return None
    return match.group(1)

unique_images["code"] = unique_images["filename"].apply(extract_code)

unmatched = unique_images[unique_images["code"].isna()]
print(f"Filenames that do NOT match the pattern: {len(unmatched)}")
for name in unmatched["filename"].head(10):
    print("  ", name)

# Split the code into its C and P parts where possible, e.g. "C33P1" -> C33, P1.
C_AND_P_PATTERN = re.compile(r"^C(\d+)P(\d+)$", re.IGNORECASE)

def split_c_and_p(code: str | None) -> tuple[str | None, str | None]:
    if code is None:
        return (None, None)
    match = C_AND_P_PATTERN.match(code)
    if match is None:
        return (None, None)
    return (f"C{match.group(1)}", f"P{match.group(2)}")

c_and_p = unique_images["code"].apply(split_c_and_p)
unique_images["c_part"] = [pair[0] for pair in c_and_p]
unique_images["p_part"] = [pair[1] for pair in c_and_p]

codes_without_c_and_p = unique_images.loc[
    unique_images["code"].notna() & unique_images["c_part"].isna(), "code"
]
print(f"\nCodes that are NOT in the 'C<number>P<number>' form: {len(codes_without_c_and_p)}")
print("Examples:", sorted(codes_without_c_and_p.unique())[:15])

## 11. Patients: how many, and how are they spread?

Questions that decide how Phase 2 splits the data:
- **How many distinct codes are there?** (Few codes = few patients = a noisy test score.)
- **Does each `C` number always come with the same `P` number?** If one `C` number appears with several `P` numbers (or the other way round), then the full code is not simply "one patient", and we must group more carefully to avoid leakage.
- **Which codes appear in both classes?** Infected patients' slides also contain healthy cells, so we expect some.

In [ ]:
print(f"Distinct full codes:  {unique_images['code'].nunique()}")
print(f"Distinct C numbers:   {unique_images['c_part'].nunique()}")
print(f"Distinct P numbers:   {unique_images['p_part'].nunique()}")

p_numbers_per_c = unique_images.dropna(subset=["c_part"]).groupby("c_part")["p_part"].nunique()
c_numbers_per_p = unique_images.dropna(subset=["p_part"]).groupby("p_part")["c_part"].nunique()
print(f"\nC numbers that appear with more than one P number: {(p_numbers_per_c > 1).sum()}")
print(f"P numbers that appear with more than one C number: {(c_numbers_per_p > 1).sum()}")

# For each code: how many cells of each class?
cells_per_code = unique_images.groupby(["code", "label"]).size().unstack(fill_value=0)
for class_name in CLASS_NAMES:
    if class_name not in cells_per_code.columns:
        cells_per_code[class_name] = 0
cells_per_code["total"] = cells_per_code["Uninfected"] + cells_per_code["Parasitized"]

has_both_classes = (cells_per_code["Uninfected"] > 0) & (cells_per_code["Parasitized"] > 0)
only_uninfected = (cells_per_code["Parasitized"] == 0)
only_parasitized = (cells_per_code["Uninfected"] == 0)
print(f"\nCodes with cells of BOTH classes:   {has_both_classes.sum()}")
print(f"Codes with ONLY Uninfected cells:   {only_uninfected.sum()}")
print(f"Codes with ONLY Parasitized cells:  {only_parasitized.sum()}")

print("\nCells per code:")
print(cells_per_code["total"].describe().round(1).to_string())

print("\nThe 5 codes with the most cells:")
print(cells_per_code.sort_values("total", ascending=False).head(5).to_string())

plt.figure(figsize=(10, 3))
plt.hist(cells_per_code["total"], bins=40)
plt.title("How many cells each code (patient/slide) contributes")
plt.xlabel("cells")
plt.ylabel("number of codes")
plt.show()

## 12. Summary (copy this for Phase 2)

Copy **everything this cell prints**. Phase 2's splitting code is built on these real facts, not on guesses.

In [ ]:
print("===== PHASE 1 SUMMARY =====")
print(f"data dir: {DATA_DIR.resolve()}")
print(f"all files: {len(all_files)} | extensions: {dict(extension_counts)}")
print(f"labelled pngs: {len(images)} | folders: {images.groupby('folder').size().to_dict()}")
print(f"unique images: {len(unique_images)} | renamed duplicates: {len(renamed_duplicates)} "
      f"| same image under both labels: {len(conflicting_hashes)}")
print(f"class counts: {unique_images['label'].value_counts().to_dict()}")
print(f"width  min/median/max: {unique_images['width'].min()}/"
      f"{int(unique_images['width'].median())}/{unique_images['width'].max()}")
print(f"height min/median/max: {unique_images['height'].min()}/"
      f"{int(unique_images['height'].median())}/{unique_images['height'].max()}")
print(f"unmatched filenames: {len(unmatched)} | examples: {list(unmatched['filename'].head(3))}")
print(f"codes not C<n>P<n>: {codes_without_c_and_p.nunique()} | examples: "
      f"{sorted(codes_without_c_and_p.unique())[:5]}")
print(f"distinct codes: {unique_images['code'].nunique()} | C numbers: "
      f"{unique_images['c_part'].nunique()} | P numbers: {unique_images['p_part'].nunique()}")
print(f"C with >1 P: {(p_numbers_per_c > 1).sum()} | P with >1 C: {(c_numbers_per_p > 1).sum()}")
print(f"codes both classes / only uninfected / only parasitized: "
      f"{has_both_classes.sum()} / {only_uninfected.sum()} / {only_parasitized.sum()}")
print(f"cells per code min/median/max: {cells_per_code['total'].min()}/"
      f"{int(cells_per_code['total'].median())}/{cells_per_code['total'].max()}")
for class_name in CLASS_NAMES:
    names = unique_images.loc[unique_images["label"] == class_name, "filename"]
    print(f"sample {class_name} filenames: {list(names.sample(n=min(3, len(names)), random_state=RANDOM_SEED))}")

## Done when

You can answer: **how many images, how many patients, what size range, and which folder quirks are real.**

### Explain it back
1. What does an infected cell look like compared to a healthy one?
2. What is patient leakage, and why would it make our test score lie?